In [ ]:
import ijson
import sumolib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pyproj import CRS, Transformer
from scipy.interpolate import LinearNDInterpolator

### Define functions for plotting ground truth time-space diagram

In [ ]:
def load_trajectories_wb(inception_file_path, trajectory_timeframe=pd.Timedelta(minutes=60), min_time=None):
        """
        Loads westbound trajectories from an Inception JSON file into a pandas DataFrame.
        Filters trajectories based on direction, time frame, and mile markers.

        Args:
            inception_file_path (str): Path to the Inception JSON file.
            trajectory_timeframe (pd.Timedelta): Time frame to load trajectories for.
            min_time (pd.Timestamp, optional): Minimum start time for trajectories.
        """
        # eastbound_trajectories = []
        westbound_trajectories = []
        t_min = None
        t_max = None
        mile_marker_start = 58.8 * 5280 * 0.3048 #meters
        mile_marker_end = 62.8 * 5280 * 0.3048 # meters
        # Open file and stream data
        with open(inception_file_path, "r") as f:
            trajectory_iterator = ijson.items(f, "item")
            
            for traj in trajectory_iterator:
                x_positions = np.array(traj.get("x_position", []), dtype=np.float32) * 0.3048  # Convert feet to meters
                y_positions = np.array(traj.get("y_position", []), dtype=np.float32) * 0.3048  # Convert feet to meters
                direction = traj.get("direction")

                if len(x_positions) > 1 and direction == -1: # -1 indicates westbound, 1 indicates eastbound
                    timestamps = np.array(traj.get("timestamp", []), dtype=np.float64)
                    timestamps = pd.to_datetime(timestamps, unit="s").astype(np.int64) / 1e9  # Convert to seconds
                    
                    if min_time and (timestamps[0] < min_time.timestamp()):
                        continue
                    
                    # eastbound_trajectories.append({
                    #     "trajectory": traj, 
                    #     "timestamps": timestamps,
                    #     "x_positions": x_positions,
                    #     "y_positions": y_positions
                    # })
                    westbound_trajectories.append({
                        "trajectory": traj, 
                        "timestamps": timestamps,
                        "x_positions": x_positions,
                        "y_positions": y_positions
                    })
                    
                    # Efficient min/max tracking
                    t_min = timestamps[0] if t_min is None else min(t_min, timestamps[0])
                    t_max = timestamps[0] if t_max is None else max(t_max, timestamps[0])

                    if t_max is not None and t_min is not None and (t_max - t_min) > trajectory_timeframe.total_seconds():
                        break 

        # print(f"Loaded {len(eastbound_trajectories)} eastbound trajectories.")
        print(f"Loaded {len(westbound_trajectories)} westbound trajectories.")

        # if not eastbound_trajectories:
        if not westbound_trajectories:
            
            return pd.DataFrame(columns=["trajectory_id", "timestamp", "x_position", "speed"])

        # Vectorized DataFrame creation
        all_trajectory_ids = []
        all_timestamps = []
        all_x_positions = []
        all_y_positions = []
        all_speeds = []

        # for idx, traj in enumerate(eastbound_trajectories):
        for idx, traj in enumerate(westbound_trajectories):
            mask = (traj["x_positions"] >= mile_marker_start) & (traj["x_positions"] <= mile_marker_end)
            filtered_timestamps = traj["timestamps"][mask]
            filtered_x_positions = traj["x_positions"][mask]
            filtered_y_positions = traj["y_positions"][mask]

            num_points = len(filtered_timestamps)
            all_trajectory_ids.extend([idx] * num_points)
            all_timestamps.extend(filtered_timestamps)
            all_x_positions.extend(filtered_x_positions)
            all_y_positions.extend(filtered_y_positions)
        df = pd.DataFrame({
            "trajectory_id": np.array(all_trajectory_ids, dtype=np.int32),
            "timestamp": pd.to_datetime(all_timestamps, unit="s"),
            "x_position": np.array(all_x_positions, dtype=np.float32),
            "y_position": np.array(all_y_positions, dtype=np.float32)
            # "speed": np.array(all_speeds, dtype=np.float32)
        })
        
        return df


def generate_time_intervals_seconds(year_str, date_str, begin_time_in_sec, end_time_in_sec, interval_len_in_sec):
    """
    Generates a list of time strings at equal intervals between a begin and end time on a given date.
    Works at the second level.

    Args:
        year_str (str): Year string in 'YYYY' format.
        date_str (str): Date string in 'MMDD' format.
        begin_time_in_sec (int): Start time in seconds from midnight (0–86399).
        end_time_in_sec (int): End time in seconds from midnight (0–86399).
        interval_len_in_sec (int): Interval length in seconds.

    Returns:
        list: A list of time strings in 'YYYY-MM-DD HH:MM:SS' format.
    """
    if interval_len_in_sec <= 0:
        return []

    # Parse date string
    date_parts = datetime.datetime.strptime(date_str, "%m%d")
    base_date = datetime.datetime(int(year_str), date_parts.month, date_parts.day)

    time_list = []
    current_time_in_sec = begin_time_in_sec

    while current_time_in_sec <= end_time_in_sec:
        current_time_obj = base_date + datetime.timedelta(seconds=current_time_in_sec)
        time_list.append(current_time_obj.strftime("%Y-%m-%d %H:%M:%S"))
        current_time_in_sec += interval_len_in_sec

    return time_list

    
def get_ground_truth_npy(df, time_interval=pd.Timedelta(seconds=10), space_interval=400):
    # Compute min/max for time and space
    
    t_min, t_max = df["timestamp"].min(), df["timestamp"].max()
    x_min, x_max = df["x_position"].min(), df["x_position"].max()

    # Ensure valid ranges
    if x_min == x_max:
        raise ValueError("x_min and x_max are identical, meaning no variation in x_position.")
    
    # Create time and space bins
    time_bins = pd.date_range(start=t_min, end=t_max, freq=time_interval)
    # print(time_bins)
    space_bins = np.arange(x_min, x_max, space_interval)
    # print(space_bins)
    
    if len(space_bins) < 2:
        raise ValueError("space_bins array is empty or too small, adjust space_interval.")
    
    # Assign bin indices using `pd.cut()`
    df["time_bin"] = pd.cut(df["timestamp"], bins=time_bins, labels=False, include_lowest=True)
    df["space_bin"] = pd.cut(df["x_position"], bins=space_bins, labels=False, include_lowest=True)

    # Remove NaNs (out-of-range values)
    df = df.dropna(subset=["time_bin", "space_bin"]).astype({"time_bin": int, "space_bin": int})

    # Compute flow and density using `groupby()`
    flow_matrix = np.zeros((len(time_bins) - 1, len(space_bins) - 1))
    density_matrix = np.zeros_like(flow_matrix)
    traj_count_matrix = np.zeros_like(flow_matrix)
    # lane_matrix = np.zeros_like(flow_matrix)

    grouped = df.groupby(["time_bin", "space_bin"])
    area_bin = (space_interval/1000.) * time_interval.total_seconds() / 3600.0 #convert space interval to kilometers, time_interval to hours
    for (time_bin, space_bin), group in grouped:
        # print(time_bin, space_bin)
        traj_group = group.groupby("trajectory_id")
        # traj_dict = {traj_id: traj_data for traj_id, traj_data in traj_group}
    
        total_distance = sum(traj_group["x_position"].apply(lambda x: x.max()-x.min()))
        total_time = sum(traj_group["timestamp"].apply(lambda x: (x.max() - x.min()).total_seconds()))

        flow_matrix[time_bin, space_bin] = (total_distance / (1000.*4)) / area_bin # Assumes 4 lanes
        density_matrix[time_bin, space_bin] = (total_time / (3600.0 * 4)) / area_bin # Assumes 4 lanes
        velocity_matrix = flow_matrix/density_matrix

        # --- Count unique trajectories in this bucket ---
        traj_count = traj_group.ngroups   # number of unique trajectory_ids
        traj_count_matrix[time_bin, space_bin] = traj_count
    
    return flow_matrix, density_matrix, velocity_matrix, traj_count_matrix

### Load, compute, and plot ground truth time-space diagram

In [ ]:
### STEP 0 -- Define variables
download_from_inception = False # if you already have the trajectory data in a csv somewhere, set this to false
calc_ground_truth_npy = False # if you already have the ground truth numpy file, set this to false
start_hour = 8
date_hyphenated = "11-30" # date of interest in MM-DD format
duration_hrs = 1


date_str = date_hyphenated.replace("-", "")
end_hour = start_hour + duration_hrs
duration = duration_hrs * 60 # in minutes

begin_time_in_seconds = start_hour * 60 * 60
end_time_in_seconds = end_hour * 60 * 60
# begin_time_in_seconds = 0
# end_time_in_seconds = 7230

# Path to load inception data from
inception_json_path = f"../i24_data/{date_hyphenated}.json"
# Path to save inception pre-processed csv to or load from
inception_csv_path = f"../i24_data/csvs/{date_str}_westbound_{start_hour}-{end_hour}.csv"

ground_truth_npy_path = f"../i24_data/npy/gt_velocity_{date_str}_westbound_{start_hour}-{end_hour}.npy"


min_time = pd.to_datetime(f"2022-{date_hyphenated} {start_hour + 6}:00:00.400000095") # The +6 is to convert from central time to utc time

# ### STEP 1 -- Load ground truth data from inception or inception pre-processed csv
# if download_from_inception:
#     ### OPTION A: Get ground truth numpy array from inception trajectory -- disregard if you already possess it ###
#     # if you want to adjust other settings such as space of interest, lane direction of interest, etc., look in the load_trajectories function itEC for now

#     trajectory_timeframe = pd.Timedelta(minutes=duration) # set based on however much time you want to load, starting from min_time
#     inception_data_df = load_trajectories_wb(inception_file_path=inception_json_path,
#                                         trajectory_timeframe=trajectory_timeframe,
#                                         min_time=min_time)

#     # # it's a good idea to save this df to csv so you don't have to read the inception data every time for future runs
#     # inception_data_df.to_csv("../i24_data/1129_eastbound_7_to_830.csv")
#     inception_data_df.to_csv(inception_csv_path)
# else:
#     # if you already have trajectory data in a csv somewhere
#     inception_data_df = pd.read_csv(inception_csv_path)
#     inception_data_df["timestamp"] = pd.to_datetime(inception_data_df["timestamp"])


### STEP 2 -- Use EC to calculate ground truth numpy array if you don't already have it
if calc_ground_truth_npy:
    # convert your df derived from the inception data file to a numpy array for plotting / error calculating
    gt_flow_npy, gt_density_npy, gt_velocity_npy, count_matrix = get_ground_truth_npy(df = inception_data_df)

    gt_velocity_npy =  np.flip(gt_velocity_npy, axis=1)
    gt_velocity_npy = gt_velocity_npy[:,:]

    # change based on your metric of interest
    ground_truth_npy_file = gt_velocity_npy

    # Save to file for future use
    np.save(ground_truth_npy_path, ground_truth_npy_file)
else:
    # load ground truth numpy
    ground_truth_npy_file = np.load(ground_truth_npy_path)

print("ground truth npy shape:", ground_truth_npy_file.shape)

### STEP 3 -- Plot
# plot the ground truth
vmax = np.max(ground_truth_npy_file)
plt.figure(figsize=(10, 6))
plt.imshow(ground_truth_npy_file.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax = 120)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title(f'Time-Space Diagram of Velocities - Ground Truth\n{date_str} Westbound {start_hour}:00 to {end_hour}:00')
plt.colorbar(label='Average Velocity (km/hr)')
plt.tight_layout()
# plt.savefig(f"path_to_i24_data/figs/gt_velocity_{date_str}_westbound_{start_hour}-{end_hour}.png", dpi=300)
plt.show()
plt.close()

#### Now extract sim-produced data and compare

In [ ]:
def to_seconds_since_midnight(ts):
        """Convert datetime string or object to seconds after midnight."""
        if isinstance(ts, str):
            ts = pd.to_datetime(ts)
        elif isinstance(ts, (pd.Timestamp,)):
            ts = ts.to_pydatetime()
        return ts.hour * 3600 + ts.minute * 60 + ts.second


def fcd_xml_to_df(fcd_xml_file_path: str, time_stamps):
        """
        Reads a SUMO FCD XML file and returns a DataFrame containing vehicle data.
        Uses iterparse for efficiency on large files.

        Args:
            fcd_xml_file_path (str): Path to the FCD XML file.
            time_stamps (list): List of datetime strings representing the time window of interest.

        Returns:
            pandas.DataFrame: A DataFrame containing all vehicle data.
        """        

        all_vehicle_data = []

        print(f"Extracting SUMO sim traj data from {fcd_xml_file_path}")

        # Efficient streaming parse
        for event, elem in ET.iterparse(fcd_xml_file_path, events=('start', 'end')):
            if event == 'start' and elem.tag == 'timestep':
                current_time = float(elem.attrib['time'])  # store timestep time
            elif event == 'end' and elem.tag == 'vehicle':
                vehicle_data = elem.attrib.copy()  # copy vehicle attributes
                vehicle_data['time'] = current_time  # add timestep time
                all_vehicle_data.append(vehicle_data)
                elem.clear()  # free memory

        if not all_vehicle_data:
            raise ValueError(f"No data found in FCD XML file: {fcd_xml_file_path}")

        # Convert to DataFrame
        df = pd.DataFrame(all_vehicle_data)

        # Convert numeric columns (using your defined list of numeric fields)
        numeric_cols = ['x', 'y', 'angle', 'speed', 'pos', 'slope', 'time', 'leaderGap']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Define start and end time
        start_time = to_seconds_since_midnight(time_stamps[0])
        end_time = to_seconds_since_midnight(time_stamps[-1])

        # Filter rows within time window
        df = df[(df['time'] >= start_time) & (df['time'] <= end_time)]

        return df


def build_sumo_xy_to_mm_mapper(mm_latlon_mapping_path, widen_half_dist_meters=40):
        """
        Builds a mapping function from SUMO (x, y) coordinates on the roadway to mile markers.
        Uses a more robust interpolation method by generating additional points offset (otherwise
        the standard linear interpolator can produce NaNs for points slightly off the road centerline).

        Parameters:
        mm_latlon_mapping_path (str): Path to the CSV file containing mile marker to lat/lon mappings.
        widen_half_dist_meters: The distance in meters to offset points on either side of the interpolation line.
                                Further out allows vehicles further from the centerline to be mapped, but too far may introduce inaccuracies.

        Returns:
            interp_func_robust: A function that takes (lon, lat) and returns the corresponding mile marker.
        """

        # --- 1. Setup Coordinate Transformation ---
        crs_wgs84 = CRS("EPSG:4326")
        crs_projected = CRS("EPSG:32616") # Nashville UTM Zone
        to_projected = Transformer.from_crs(crs_wgs84, crs_projected, always_xy=True)
        to_wgs84 = Transformer.from_crs(crs_projected, crs_wgs84, always_xy=True)

        # --- 2. Convert original points to meters ---
        mm_latlon_df = pd.read_csv(mm_latlon_mapping_path)
        points_df = mm_latlon_df # Just create an alias for easier reference
        lons = points_df['X_WGS84'].values
        lats = points_df['Y_WGS84'].values
        x_meter, y_meter = to_projected.transform(lons, lats)

        # --- 3. Generate Widened Points and Store Intermediate Values ---

        # These lists are for building the interpolator
        wide_points_meter = []
        all_mm_values = []

        for i in range(len(x_meter)):
            # Add original point for interpolator
            wide_points_meter.append((x_meter[i], y_meter[i]))
            all_mm_values.append(points_df['MM'].iloc[i])

            # Calculate vector for perpendicular offset 
            if i < len(x_meter) - 1:
                vec_x, vec_y = x_meter[i+1] - x_meter[i], y_meter[i+1] - y_meter[i]
            else:
                vec_x, vec_y = x_meter[i] - x_meter[i-1], y_meter[i] - y_meter[i-1]
            norm = np.sqrt(vec_x**2 + vec_y**2)
            perp_x, perp_y = -vec_y / norm, vec_x / norm

            # Calculate meter offsets and the new points
            dx1, dy1 = perp_x * widen_half_dist_meters, perp_y * widen_half_dist_meters
            dx2, dy2 = -perp_x * widen_half_dist_meters, -perp_y * widen_half_dist_meters
            new_x1, new_y1 = x_meter[i] + dx1, y_meter[i] + dy1
            new_x2, new_y2 = x_meter[i] + dx2, y_meter[i] + dy2

            # Add offset points for the interpolator
            wide_points_meter.append((new_x1, new_y1))
            wide_points_meter.append((new_x2, new_y2))
            all_mm_values.append(points_df['MM'].iloc[i])
            all_mm_values.append(points_df['MM'].iloc[i])

        # --- 5. Build and Return the Robust Interpolator  ---
        wide_points_meter_arr = np.array(wide_points_meter)
        wide_lons, wide_lats = to_wgs84.transform(wide_points_meter_arr[:, 0], wide_points_meter_arr[:, 1])
        wide_lonlat_points = np.vstack((wide_lons, wide_lats)).T
        interp_func_robust = LinearNDInterpolator(wide_lonlat_points, all_mm_values)

        return interp_func_robust


def get_route_edge_ids(net, start_edge_id, end_edge_id):
        """
        Function to get an ordered list of edge IDs along each route.
        
        Args:
            net (sumolib.net.Net): The SUMO network object.
            start_edge_id (str): The ID of the starting edge.
            end_edge_id (str): The ID of the ending edge.
        
        Returns:
            list: An ordered list of edge IDs representing the route from start to end.
        """
        
        route, _ = net.getShortestPath(
            net.getEdge(start_edge_id), 
            net.getEdge(end_edge_id)
        )

        edge_ids = [edge.getID() for edge in route]

        return edge_ids


def compute_velocity_grid_from_fcd(
        fcd_xml_file_path,
        net_file_path,
        mm_latlon_mapping_path,
        time_stamps,
        eb_route_start_edge_id,
        eb_route_end_edge_id,
        wb_route_start_edge_id,
        wb_route_end_edge_id,
        space_interval=400,     # in meters
        x_min_miles=58.8,
        x_max_miles=62.8,
        return_df=False,
    ):
        """
        Build a 2D grid of average velocities from SUMO FCD data.
        Rows = time bins, Cols = space bins.

        Args:
            fcd_xml_file_path (str): Path to the FCD XML file.
            net_file_path (str): Path to the SUMO network file (.net.xml).
            mm_latlon_mapping_path (str): Path to the CSV file containing mile marker to lat/lon mappings.
            time_stamps (list): List of datetime strings representing the time window of interest.
            eb_route_start_edge_id (str): Starting edge ID for the eastbound route.
            eb_route_end_edge_id (str): Ending edge ID for the eastbound route.
            wb_route_start_edge_id (str): Starting edge ID for the westbound route.
            wb_route_end_edge_id (str): Ending edge ID for the westbound route.
            space_interval (float): Width of each space bin (meters).
            x_min_miles (float): Lower bound of road segment in miles.
            x_max_miles (float): Upper bound of road segment in miles.
            return_df (bool): If True, also return the pivoted DataFrame for inspection.
        """

        # --- Load FCD file ---
        df = fcd_xml_to_df(fcd_xml_file_path, time_stamps)
        print(f"FCD DataFrame shape (before filtering): {df.shape}")

        # Extract edge_id (direction indicator)
        df['edge_id'] = df['lane'].apply(lambda x: x.split('_')[0])

        # --- Build interpolator for mile marker mapping ---
        interp_func_robust = build_sumo_xy_to_mm_mapper(mm_latlon_mapping_path)

        # --- Map SUMO (x,y) → lon/lat → mile marker ---
        net = sumolib.net.readNet(net_file_path, withInternal=True)
        lonlat = np.array([net.convertXY2LonLat(x, y) for x, y in zip(df['x'], df['y'])])
        df['mm'] = interp_func_robust(lonlat)
        df = df.dropna(subset=['mm']) # Drop rows where mm is NaN; these points are beyond the convex hull of the mile marker data (likely too far up/down road or off to the side)
        print(f"FCD DataFrame shape (after mm mapping & filtering): {df.shape}")

        # --- Filter to road segment ---
        df = df[(df['mm'] >= x_min_miles) & (df['mm'] <= x_max_miles)]
        print(f"FCD DataFrame shape (after segment filter): {df.shape}")

        if df.empty:
            raise ValueError("No FCD data in the specified segment and time range.")

        # --- Setup time bins based on micro_obj_function_timestamps ---
        time_bins = np.array([to_seconds_since_midnight(ts) for ts in time_stamps],dtype=float)

        n_expected_bins = len(time_bins)
        print(f"Using {n_expected_bins} time bins from micro_obj_function_timestamps")

        # --- Setup space bins ---
        x_min = x_min_miles * 1609.34
        x_max = x_max_miles * 1609.34
        space_bins = np.arange(x_min, x_max, space_interval)

            # --- Helper function to compute velocity grid for one direction ---
        def _build_velocity_grid(direction_ids, direction_label):
            sub_df = df[df['edge_id'].isin(direction_ids)]
            print(f"{direction_label} DataFrame shape: {sub_df.shape}")

            if sub_df.empty:
                print(f"Warning: no {direction_label} data found within this segment.")
                pivot_table = pd.DataFrame(np.nan, index=range(n_expected_bins), columns=range(len(space_bins)-1))
            else:
                # Assign bins
                sub_df["time_bin"] = pd.cut(sub_df["time"], bins=time_bins, labels=False, include_lowest=True)
                sub_df["space_bin"] = pd.cut(sub_df["mm"] * 1609.34, bins=space_bins, labels=False, include_lowest=True)
                sub_df = sub_df.dropna(subset=["time_bin", "space_bin"]).astype({"time_bin": int, "space_bin": int})

                # Pivot to grid
                pivot_table = sub_df.pivot_table(
                    index="time_bin",
                    columns="space_bin",
                    values="speed",
                    aggfunc="mean"
                ).reindex(
                    index=range(n_expected_bins),
                    columns=range(len(space_bins) - 1)
                )

                # Convert m/s → km/h
                pivot_table = pivot_table * 3.6

            if direction_label == "WB":
                # Flip horizontally
                velocity_grid = np.flip(pivot_table.to_numpy(), axis=1)

            else:
                velocity_grid = pivot_table.to_numpy()

            return velocity_grid, pivot_table

        # Get the full extent of the eb and wb routes
        eb_route_edge_ids = get_route_edge_ids(
            net=net,
            start_edge_id=eb_route_start_edge_id,
            end_edge_id=eb_route_end_edge_id,
        )
        wb_route_edge_ids = get_route_edge_ids(
            net=net,
            start_edge_id=wb_route_start_edge_id,
            end_edge_id=wb_route_end_edge_id,
        )

        # --- Compute both directions ---
        wb_velocity_grid, wb_pivot_table = _build_velocity_grid(wb_route_edge_ids, "WB")
        eb_velocity_grid, eb_pivot_table = _build_velocity_grid(eb_route_edge_ids, "EB")

        # --- Return according to flag ---
        if return_df:
            return eb_velocity_grid, eb_pivot_table, wb_velocity_grid, wb_pivot_table
        else:
            return eb_velocity_grid, wb_velocity_grid

In [ ]:
## STEP 2 -- Initialize EC
fcd_xml_file_path = "path_to_sim_outputs/fcd_output.xml"
net_file_path = "path_to_sim_files/sumo.net.xml"
mm_latlon_mapping_path = "path_to_sim_files/mile_marker_layer.csv"

year_str = "2022"
interval = 10 #seconds

# Generate time stamps for the period of interest.
time_stamps = generate_time_intervals_seconds(year_str, date_str, begin_time_in_seconds, end_time_in_seconds, interval)
time_stamps = time_stamps[:-1]

# Compute the velocity grid from the FCD data.
eb_npy, eb_df, wb_npy, wb_df = compute_velocity_grid_from_fcd(
    fcd_xml_file_path,
    net_file_path,
    mm_latlon_mapping_path,
    time_stamps,
    eb_route_start_edge_id='108162303',
    eb_route_end_edge_id='988591740#0',
    wb_route_start_edge_id='974949114',
    wb_route_end_edge_id='634155175',
    return_df=True
)
print(eb_npy.shape)
print(wb_npy.shape)
# glimpse at the velocity grid
wb_df.head()



In [ ]:
# plot the FCD-derived velocity
plt.figure(figsize=(10, 6))
plt.imshow(wb_npy.T, aspect='auto', origin='lower', cmap='RdYlGn', interpolation="none", vmin = 0, vmax=vmax)
# Label axes
plt.xlabel('Time Step')
plt.ylabel('Segment Index')
plt.title('Time-Space Diagram of Velocities - Simulation-Generated')
plt.colorbar(label='Velocity (km/hr)')
plt.tight_layout()

In [ ]:
# Set consistent vmax across both plots so colors are comparable
vmax = max(np.max(ground_truth_npy_file), np.max(wb_npy))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# --- Ground Truth ---
im0 = axes[0].imshow(
    ground_truth_npy_file.T,
    aspect='auto',
    origin='lower',
    cmap='RdYlGn',
    interpolation="none",
    vmin=0,
    vmax=vmax
)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Segment Index')
axes[0].set_title('Velocity Ground Truth')

# --- Predicted ---
im1 = axes[1].imshow(
    wb_npy.T,
    aspect='auto',
    origin='lower',
    cmap='RdYlGn',
    interpolation="none",
    vmin=0,
    vmax=vmax
)
axes[1].set_xlabel('Time Step')
axes[1].set_title('Predicted Velocity')

# Add a single colorbar for both plots
fig.colorbar(im1, ax=axes, orientation='vertical', label='Velocity (km/hr)')

plt.show()